# 02 - ETL: CSV → Parquet → BigQuery
## Home Credit Default Risk · Credit Scoring Model

Notebook ini membangun **pipeline data** agar data mentah (8 file CSV, ±2,6 GB) tersimpan rapi, cepat dibaca, dan dapat diakses oleh notebook analisis, modeling, dan dashboard Power BI.

## 1. Kenapa perlu ETL?

**Masalah tanpa ETL:**
- Data terdiri dari **8 tabel** yang saling berhubungan. Satu nasabah bisa punya puluhan riwayat pinjaman dan ratusan catatan cicilan (*one-to-many*).
- Setiap notebook harus membaca ulang CSV berukuran besar dan menghitung ulang agregasinya. Prosesnya **lambat, berulang, dan rawan berbeda hasil** antar notebook.

**Solusi:** pipeline yang dijalankan **sekali**, lalu hasilnya dipakai bersama:

| Tahap | Yang dilakukan | Hasil |
|---|---|---|
| **Extract** | Membaca 8 file CSV mentah | Data di memori |
| **Transform (ringan)** | Memperbaiki tipe data & menghemat memori | Data siap simpan |
| **Load** | Menyimpan ke **Parquet** (lokal) lalu ke **BigQuery** (cloud) | Data tersimpan rapi |
| **Transform (di BigQuery)** | Agregasi tabel pendukung jadi 1 baris per nasabah memakai SQL, lalu digabung ke tabel utama | **Data mart** siap analisis |

> Pembersihan anomali (misalnya `DAYS_EMPLOYED = 365243`) **tidak** dilakukan di sini, melainkan di NB 03 setelah data quality check, agar data di warehouse tetap **asli dan dapat ditelusuri**.

## 2. Apa itu Parquet dan kenapa dipakai?

**Analogi:**
- **CSV** seperti buku catatan tulisan tangan: semua orang bisa membaca, tapi tebal dan lambat dicari.
- **Parquet** seperti arsip yang dipadatkan dan diberi indeks: kecil dan cepat, tapi dibuka dengan alat khusus (pandas, Power BI, BigQuery).

| Aspek | CSV | Parquet |
|---|---|---|
| Cara simpan | Per **baris**, teks biasa | Per **kolom**, biner & terkompresi |
| Ukuran | Besar | Umumnya beberapa kali lebih kecil |
| Kecepatan baca | Lambat (harus parsing teks) | Cepat, bisa baca **kolom tertentu saja** |
| Tipe data | Hilang (semua teks), ditebak ulang setiap dibaca | Tersimpan (angka tetap angka) |
| Dibuka di Excel | Bisa | Tidak bisa langsung |

**Peran Parquet di proyek ini = tempat transit (staging):**
1. **Upload ke BigQuery lebih cepat**, karena file lebih kecil dan tipe data sudah benar.
2. **Backup lokal**: tabel di BigQuery Sandbox terhapus otomatis setelah 60 hari, dan Parquet bisa di-upload ulang kapan saja.
3. **Bisa dipakai tanpa internet** untuk analisis dan modeling.

**Arsitektur:**
```
data/raw/*.csv ──► Extract & Transform ringan ──► data/staging/*.parquet
                                                     │ Load
                                                     ▼
                                    BigQuery dataset `home_credit_raw`   (8 tabel asli)
                                                     │ Transform (SQL)
                                                     ▼
                                    BigQuery dataset `home_credit_mart`  (agregasi + mart_train / mart_test)
                                                     │
                     ┌───────────────────────────────┼─────────────────────────┐
              NB 03 EDA                     Power BI dashboard          NB 04 Modeling
                                                     │
                                         data/processed/*.parquet (salinan lokal mart)
```

## 3. Persiapan BigQuery (dilakukan sekali)

1. Buka https://console.cloud.google.com → login akun Google → **buat project baru** (contoh: `home-credit-risk`). Catat **Project ID**-nya.
2. Buka menu **BigQuery**. Jika belum ada billing, otomatis memakai **BigQuery Sandbox** (gratis: storage 10 GB, query 1 TB/bulan, tabel kedaluwarsa 60 hari).
3. Menu **IAM & Admin → Service Accounts → Create service account**
   - Nama: `etl-home-credit`
   - Role: **BigQuery Admin**
4. Buka service account tersebut → tab **Keys → Add key → Create new key → JSON** → file terunduh.
5. Pindahkan file JSON ke folder proyek, ganti nama menjadi `gcp-key.json`.
6. **Jangan push file key ke GitHub.** File ini sudah dimasukkan ke `.gitignore`.

## 4. Setup

In [1]:
# Jalankan sekali jika library belum terpasang
# !pip install pandas pyarrow google-cloud-bigquery db-dtypes

In [2]:
import os, time, warnings, logging
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

# Sembunyikan warning agar output rapi
warnings.filterwarnings('ignore')
logging.getLogger('google').setLevel(logging.ERROR)

# ===== KONFIGURASI =====
RAW_DIR       = Path('../home-credit-default-risk')   # folder CSV mentah
STAGING_DIR   = Path('data/staging')                   # hasil Parquet
PROCESSED_DIR = Path('data/processed')                 # salinan lokal data mart
KEY_PATH      = 'gcp-key.json'                         # key service account
PROJECT_ID    = 'home-credit-risk-511109'              # Project ID BigQuery
DATASET_RAW   = 'home_credit_raw'
DATASET_MART  = 'home_credit_mart'
LOCATION      = 'US'                                   # lokasi dataset BigQuery

TABLES = ['application_train', 'application_test', 'bureau', 'bureau_balance',
          'previous_application', 'POS_CASH_balance', 'credit_card_balance', 'installments_payments']

STAGING_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 5. Extract: inventaris data mentah

In [3]:
inventory = pd.DataFrame([{'tabel': t, 'ukuran_csv_MB': round(os.path.getsize(RAW_DIR / f'{t}.csv') / 1e6, 1)}
                          for t in TABLES])
print(f"Total ukuran CSV: {inventory.ukuran_csv_MB.sum()/1000:.2f} GB")
inventory

Total ukuran CSV: 2.68 GB


,tabel,ukuran_csv_MB
0,application_train,166.1
1,application_test,26.6
2,bureau,170.0
3,bureau_balance,375.6
4,previous_application,405.0
5,POS_CASH_balance,392.7
6,credit_card_balance,424.6
7,installments_payments,723.1


## 6. Transform ringan → simpan Parquet

Transformasi yang dilakukan hanya **teknis**, tanpa mengubah isi data:
1. **Downcast tipe numerik** (misal `float64 → float32`, `int64 → int32`) untuk menghemat memori.
2. Kolom teks disimpan sebagai **string**.
3. Nama kolom dirapikan (spasi di awal/akhir dihapus).

In [4]:
def optimize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = df.columns.str.strip()
    for col in df.select_dtypes('integer').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes('float').columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(['object', 'string']).columns:
        df[col] = df[col].astype('string')
    return df

def csv_to_parquet(table: str) -> dict:
    t0 = time.time()
    df = pd.read_csv(RAW_DIR / f'{table}.csv', low_memory=False)
    mem_before = df.memory_usage(deep=True).sum() / 1e6
    df = optimize_dtypes(df)
    mem_after = df.memory_usage(deep=True).sum() / 1e6
    out = STAGING_DIR / f'{table}.parquet'
    df.to_parquet(out, index=False, compression='snappy')
    return {'tabel': table, 'baris': len(df), 'kolom': df.shape[1],
            'memori_awal_MB': round(mem_before), 'memori_setelah_MB': round(mem_after),
            'ukuran_csv_MB': round(os.path.getsize(RAW_DIR / f'{table}.csv') / 1e6, 1),
            'ukuran_parquet_MB': round(os.path.getsize(out) / 1e6, 1),
            'durasi_detik': round(time.time() - t0, 1)}

# ===== Jalankan konversi: hanya untuk CSV yang baru / berubah =====
FORCE_REBUILD_PARQUET = False                    # True = buat ulang semua
LOG_PATH = STAGING_DIR / '_etl_log.csv'          # catatan hasil konversi terakhir

log = pd.read_csv(LOG_PATH).set_index('tabel') if LOG_PATH.exists() else pd.DataFrame()

def perlu_diproses(table: str) -> bool:
    src, out = RAW_DIR / f'{table}.csv', STAGING_DIR / f'{table}.parquet'
    if FORCE_REBUILD_PARQUET or not out.exists() or table not in log.index:
        return True
    return src.stat().st_mtime > out.stat().st_mtime   # CSV lebih baru dari Parquet?

rows = []
for t in TABLES:
    if perlu_diproses(t):
        res = csv_to_parquet(t)
        res['status'] = 'diproses'
        print(f"OK {t:<25} {res['baris']:>12,} baris  {res['durasi_detik']:>6}s")
    else:
        res = log.loc[t].to_dict(); res['tabel'] = t
        res['status'] = 'tidak berubah'
        print(f"SKIP {t:<25} tidak berubah, memakai hasil sebelumnya")
    rows.append(res)

summary = pd.DataFrame(rows)
summary['hemat_ukuran_%'] = (100 - summary.ukuran_parquet_MB / summary.ukuran_csv_MB * 100).round(1)
summary.drop(columns='status').to_csv(LOG_PATH, index=False)   # perbarui log

cols = ['tabel', 'status', 'baris', 'kolom', 'memori_awal_MB', 'memori_setelah_MB',
        'ukuran_csv_MB', 'ukuran_parquet_MB', 'hemat_ukuran_%', 'durasi_detik']
display(summary[cols])
print(f"Total CSV: {summary.ukuran_csv_MB.sum():,.0f} MB → Parquet: {summary.ukuran_parquet_MB.sum():,.0f} MB")

SKIP application_train         tidak berubah, memakai hasil sebelumnya
SKIP application_test          tidak berubah, memakai hasil sebelumnya
SKIP bureau                    tidak berubah, memakai hasil sebelumnya
SKIP bureau_balance            tidak berubah, memakai hasil sebelumnya
SKIP previous_application      tidak berubah, memakai hasil sebelumnya
SKIP POS_CASH_balance          tidak berubah, memakai hasil sebelumnya
SKIP credit_card_balance       tidak berubah, memakai hasil sebelumnya
SKIP installments_payments     tidak berubah, memakai hasil sebelumnya


,tabel,status,baris,kolom,memori_awal_MB,memori_setelah_MB,ukuran_csv_MB,ukuran_parquet_MB,hemat_ukuran_%,durasi_detik
0,application_train,tidak berubah,307511.0,122.0,341.0,176.0,166.1,21.6,87.0,6.2
1,application_test,tidak berubah,48744.0,121.0,54.0,28.0,26.6,3.9,85.3,0.8
2,bureau,tidak berubah,1716428.0,17.0,285.0,218.0,170.0,36.4,78.6,4.0
3,bureau_balance,tidak berubah,27299925.0,3.0,682.0,382.0,375.6,16.5,95.6,11.2
4,previous_application,tidak berubah,1670214.0,37.0,704.0,584.0,405.0,60.0,85.2,9.7
5,POS_CASH_balance,tidak berubah,10001358.0,8.0,702.0,352.0,392.7,105.3,73.2,10.5
6,credit_card_balance,tidak berubah,3840312.0,23.0,730.0,527.0,424.6,113.6,73.2,10.3
7,installments_payments,tidak berubah,13605401.0,8.0,871.0,517.0,723.1,281.7,61.0,15.2


Total CSV: 2,684 MB → Parquet: 639 MB


### 6.1 Validasi: jumlah baris CSV = Parquet

In [5]:
def count_csv_rows(path):
    with open(path, 'rb') as f:
        return sum(1 for _ in f) - 1   # dikurangi header

valid = []
for t in TABLES:
    n_csv = count_csv_rows(RAW_DIR / f'{t}.csv')
    n_pq  = pq.ParquetFile(STAGING_DIR / f'{t}.parquet').metadata.num_rows
    valid.append({'tabel': t, 'baris_csv': n_csv, 'baris_parquet': n_pq, 'cocok': n_csv == n_pq})
valid = pd.DataFrame(valid); display(valid)
assert valid.cocok.all(), 'Ada tabel yang jumlah barisnya tidak cocok!'

,tabel,baris_csv,baris_parquet,cocok
0,application_train,307511,307511,True
1,application_test,48744,48744,True
2,bureau,1716428,1716428,True
3,bureau_balance,27299925,27299925,True
4,previous_application,1670214,1670214,True
5,POS_CASH_balance,10001358,10001358,True
6,credit_card_balance,3840312,3840312,True
7,installments_payments,13605401,13605401,True


### 6.2 Bukti manfaat: kecepatan baca CSV vs Parquet

In [6]:
t0 = time.time(); pd.read_csv(RAW_DIR / 'installments_payments.csv'); t_csv = time.time() - t0
t0 = time.time(); pd.read_parquet(STAGING_DIR / 'installments_payments.parquet'); t_pq = time.time() - t0
print(f'installments_payments  CSV: {t_csv:.1f}s  |  Parquet: {t_pq:.1f}s  →  {t_csv/t_pq:.0f}x lebih cepat')

installments_payments  CSV: 7.2s  |  Parquet: 0.6s  →  12x lebih cepat


## 7. Load: Parquet → BigQuery (dataset `home_credit_raw`)

In [7]:
from google.cloud import bigquery
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(KEY_PATH)
client = bigquery.Client(project=PROJECT_ID, credentials=credentials, location=LOCATION)

for ds in [DATASET_RAW, DATASET_MART]:
    client.create_dataset(bigquery.Dataset(f'{PROJECT_ID}.{ds}'), exists_ok=True)
    print(f'OK dataset {ds} siap')

OK dataset home_credit_raw siap
OK dataset home_credit_mart siap


In [8]:
from google.api_core.exceptions import NotFound

FORCE_RELOAD_BQ = False   # True = upload ulang semua (hemat kuota: pakai hanya jika perlu)

job_config = bigquery.LoadJobConfig(source_format=bigquery.SourceFormat.PARQUET,
                                    write_disposition='WRITE_TRUNCATE')   # timpa, tidak menumpuk

load_rows = []
for t in TABLES:
    table_id  = f'{PROJECT_ID}.{DATASET_RAW}.{t.lower()}'
    pq_path   = STAGING_DIR / f'{t}.parquet'
    n_parquet = pq.ParquetFile(pq_path).metadata.num_rows
    try:
        tbl = client.get_table(table_id)
        n_bq, bq_modified = tbl.num_rows, tbl.modified.timestamp()
    except NotFound:
        n_bq, bq_modified = None, 0

    # Upload jika: tabel belum ada, jumlah baris beda, atau Parquet lebih baru dari tabel di BigQuery
    perlu_upload = FORCE_RELOAD_BQ or n_bq != n_parquet or pq_path.stat().st_mtime > bq_modified

    t0 = time.time()
    if perlu_upload:
        with open(pq_path, 'rb') as f:
            client.load_table_from_file(f, table_id, job_config=job_config).result()
        n_bq, status = client.get_table(table_id).num_rows, 'di-upload'
        print(f'OK {t:<25} {n_bq:>12,} baris  di-upload  {time.time()-t0:.0f}s')
    else:
        status = 'tidak berubah'
        print(f'SKIP {t:<25} {n_bq:>12,} baris  tidak berubah, dilewati')

    load_rows.append({'tabel': t, 'status': status, 'baris_parquet': n_parquet,
                      'baris_bigquery': n_bq, 'cocok': n_parquet == n_bq})

load_summary = pd.DataFrame(load_rows)
display(load_summary)
assert load_summary.cocok.all(), 'Ada tabel yang jumlah barisnya tidak cocok!'

SKIP application_train              307,511 baris  tidak berubah, dilewati
SKIP application_test                48,744 baris  tidak berubah, dilewati
SKIP bureau                       1,716,428 baris  tidak berubah, dilewati
SKIP bureau_balance              27,299,925 baris  tidak berubah, dilewati
SKIP previous_application         1,670,214 baris  tidak berubah, dilewati
SKIP POS_CASH_balance            10,001,358 baris  tidak berubah, dilewati
SKIP credit_card_balance          3,840,312 baris  tidak berubah, dilewati
SKIP installments_payments       13,605,401 baris  tidak berubah, dilewati


,tabel,status,baris_parquet,baris_bigquery,cocok
0,application_train,tidak berubah,307511,307511,True
1,application_test,tidak berubah,48744,48744,True
2,bureau,tidak berubah,1716428,1716428,True
3,bureau_balance,tidak berubah,27299925,27299925,True
4,previous_application,tidak berubah,1670214,1670214,True
5,POS_CASH_balance,tidak berubah,10001358,10001358,True
6,credit_card_balance,tidak berubah,3840312,3840312,True
7,installments_payments,tidak berubah,13605401,13605401,True


### 7.1 Validasi: jumlah baris Parquet = BigQuery

In [9]:
bq_check = pd.DataFrame([{'tabel': t,
                          'baris_parquet': pq.ParquetFile(STAGING_DIR / f'{t}.parquet').metadata.num_rows,
                          'baris_bigquery': client.get_table(f'{PROJECT_ID}.{DATASET_RAW}.{t.lower()}').num_rows}
                         for t in TABLES])
bq_check['cocok'] = bq_check.baris_parquet == bq_check.baris_bigquery
display(bq_check); assert bq_check.cocok.all()

,tabel,baris_parquet,baris_bigquery,cocok
0,application_train,307511,307511,True
1,application_test,48744,48744,True
2,bureau,1716428,1716428,True
3,bureau_balance,27299925,27299925,True
4,previous_application,1670214,1670214,True
5,POS_CASH_balance,10001358,10001358,True
6,credit_card_balance,3840312,3840312,True
7,installments_payments,13605401,13605401,True


## 8. Transform di BigQuery: membangun Data Mart

Tabel pendukung bersifat *one-to-many*, sehingga diringkas menjadi **1 baris per nasabah (`SK_ID_CURR`)** memakai SQL. Fitur yang dibuat dipilih berdasarkan **hipotesis H6** (riwayat kredit & perilaku bayar) di NB 01.

| Tabel agregasi | Sumber | Contoh fitur |
|---|---|---|
| `agg_bureau` | bureau + bureau_balance | jumlah kredit, kredit aktif, total utang, total tunggakan, status keterlambatan terburuk |
| `agg_prev` | previous_application | jumlah pengajuan, % ditolak, % disetujui, rata-rata pinjaman |
| `agg_installments` | installments_payments | rata-rata hari telat, % cicilan telat, rasio pembayaran |
| `agg_pos` | POS_CASH_balance | maksimum DPD, % bulan menunggak |
| `agg_cc` | credit_card_balance | utilisasi limit, maksimum DPD |

In [10]:
def T(dataset, name):
    return f"`{PROJECT_ID}.{dataset}.{name}`"
R = lambda name: T(DATASET_RAW, name)

SQL_AGG = {
'agg_bureau': f"""
WITH bb AS (
  SELECT SK_ID_BUREAU,
         MAX(CASE STATUS WHEN '1' THEN 1 WHEN '2' THEN 2 WHEN '3' THEN 3
                         WHEN '4' THEN 4 WHEN '5' THEN 5 ELSE 0 END) AS worst_status
  FROM {R('bureau_balance')} GROUP BY SK_ID_BUREAU)
SELECT b.SK_ID_CURR,
       COUNT(*)                                                    AS BUREAU_CNT,
       SUM(CASE WHEN b.CREDIT_ACTIVE = 'Active' THEN 1 ELSE 0 END) AS BUREAU_ACTIVE_CNT,
       SUM(b.AMT_CREDIT_SUM_DEBT)                                  AS BUREAU_DEBT_SUM,
       SUM(b.AMT_CREDIT_SUM_OVERDUE)                               AS BUREAU_OVERDUE_SUM,
       MAX(b.CREDIT_DAY_OVERDUE)                                   AS BUREAU_DAY_OVERDUE_MAX,
       AVG(b.DAYS_CREDIT)                                          AS BUREAU_DAYS_CREDIT_MEAN,
       MAX(bb.worst_status)                                        AS BUREAU_WORST_STATUS
FROM {R('bureau')} b LEFT JOIN bb ON b.SK_ID_BUREAU = bb.SK_ID_BUREAU
GROUP BY b.SK_ID_CURR""",

'agg_prev': f"""
SELECT SK_ID_CURR,
       COUNT(*)                                                             AS PREV_CNT,
       AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Refused'  THEN 1 ELSE 0 END)   AS PREV_REFUSED_RATE,
       AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1 ELSE 0 END)   AS PREV_APPROVED_RATE,
       AVG(AMT_CREDIT)                                                      AS PREV_AMT_CREDIT_MEAN,
       AVG(AMT_APPLICATION / NULLIF(AMT_CREDIT, 0))                          AS PREV_APP_CREDIT_RATIO_MEAN,
       AVG(CNT_PAYMENT)                                                     AS PREV_CNT_PAYMENT_MEAN
FROM {R('previous_application')}
GROUP BY SK_ID_CURR""",

'agg_installments': f"""
SELECT SK_ID_CURR,
       COUNT(*)                                                                    AS INST_CNT,
       AVG(GREATEST(DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT, 0))                      AS INST_DAYS_LATE_MEAN,
       MAX(DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT)                                   AS INST_DAYS_LATE_MAX,
       AVG(CASE WHEN DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT THEN 1 ELSE 0 END)       AS INST_LATE_RATE,
       AVG(AMT_PAYMENT / NULLIF(AMT_INSTALMENT, 0))                                AS INST_PAYMENT_RATIO_MEAN
FROM {R('installments_payments')}
GROUP BY SK_ID_CURR""",

'agg_pos': f"""
SELECT SK_ID_CURR,
       COUNT(*)                                       AS POS_MONTHS_CNT,
       MAX(SK_DPD)                                    AS POS_DPD_MAX,
       AVG(CASE WHEN SK_DPD > 0 THEN 1 ELSE 0 END)    AS POS_DPD_RATE
FROM {R('pos_cash_balance')}
GROUP BY SK_ID_CURR""",

'agg_cc': f"""
SELECT SK_ID_CURR,
       COUNT(*)                                                        AS CC_MONTHS_CNT,
       AVG(AMT_BALANCE / NULLIF(AMT_CREDIT_LIMIT_ACTUAL, 0))           AS CC_UTILIZATION_MEAN,
       MAX(SK_DPD)                                                     AS CC_DPD_MAX
FROM {R('credit_card_balance')}
GROUP BY SK_ID_CURR""",
}

def build_mart(app_table):
    M = lambda name: T(DATASET_MART, name)
    return f"""
SELECT a.*,
       b.* EXCEPT(SK_ID_CURR), p.* EXCEPT(SK_ID_CURR), i.* EXCEPT(SK_ID_CURR),
       pc.* EXCEPT(SK_ID_CURR), c.* EXCEPT(SK_ID_CURR)
FROM {R(app_table)} a
LEFT JOIN {M('agg_bureau')}       b  USING (SK_ID_CURR)
LEFT JOIN {M('agg_prev')}         p  USING (SK_ID_CURR)
LEFT JOIN {M('agg_installments')} i  USING (SK_ID_CURR)
LEFT JOIN {M('agg_pos')}          pc USING (SK_ID_CURR)
LEFT JOIN {M('agg_cc')}           c  USING (SK_ID_CURR)"""

In [11]:
def run_to_table(sql, table_name):
    t0 = time.time()
    dest = f'{PROJECT_ID}.{DATASET_MART}.{table_name}'
    cfg = bigquery.QueryJobConfig(destination=dest, write_disposition='WRITE_TRUNCATE')
    job = client.query(sql, job_config=cfg); job.result()
    print(f'OK {table_name:<18} {client.get_table(dest).num_rows:>10,} baris | '
          f'{job.total_bytes_processed/1e6:,.0f} MB diproses | {time.time()-t0:.0f}s')

for name, sql in SQL_AGG.items():
    run_to_table(sql, name)
run_to_table(build_mart('application_train'), 'mart_train')
run_to_table(build_mart('application_test'),  'mart_test')

OK agg_bureau            305,811 baris | 394 MB diproses | 7s
OK agg_prev              338,857 baris | 67 MB diproses | 7s
OK agg_installments      339,587 baris | 544 MB diproses | 6s
OK agg_pos               337,252 baris | 160 MB diproses | 23s
OK agg_cc                103,558 baris | 123 MB diproses | 5s
OK mart_train            307,511 baris | 310 MB diproses | 13s
OK mart_test              48,744 baris | 106 MB diproses | 9s


### 8.1 Validasi Data Mart

In [12]:
checks = client.query(f'''
SELECT 'mart_train' AS tabel, COUNT(*) AS baris, COUNT(DISTINCT SK_ID_CURR) AS nasabah_unik
FROM {T(DATASET_MART, 'mart_train')}
UNION ALL
SELECT 'mart_test', COUNT(*), COUNT(DISTINCT SK_ID_CURR) FROM {T(DATASET_MART, 'mart_test')}
''').to_dataframe()
checks['1_baris_per_nasabah'] = checks.baris == checks.nasabah_unik
display(checks)
assert checks['1_baris_per_nasabah'].all()
assert checks.loc[checks.tabel == 'mart_train', 'baris'].item() == client.get_table(f'{PROJECT_ID}.{DATASET_RAW}.application_train').num_rows

,tabel,baris,nasabah_unik,1_baris_per_nasabah
0,mart_train,307511,307511,True
1,mart_test,48744,48744,True


In [13]:
# Cakupan: berapa % nasabah yang punya riwayat di tiap tabel pendukung
client.query(f'''
SELECT COUNT(*) AS total,
       ROUND(100 * COUNTIF(BUREAU_CNT IS NOT NULL) / COUNT(*), 1) AS pct_ada_bureau,
       ROUND(100 * COUNTIF(PREV_CNT   IS NOT NULL) / COUNT(*), 1) AS pct_ada_prev,
       ROUND(100 * COUNTIF(INST_CNT   IS NOT NULL) / COUNT(*), 1) AS pct_ada_installments,
       ROUND(100 * COUNTIF(POS_MONTHS_CNT IS NOT NULL) / COUNT(*), 1) AS pct_ada_pos,
       ROUND(100 * COUNTIF(CC_MONTHS_CNT  IS NOT NULL) / COUNT(*), 1) AS pct_ada_cc
FROM {T(DATASET_MART, 'mart_train')}''').to_dataframe()

,total,pct_ada_bureau,pct_ada_prev,pct_ada_installments,pct_ada_pos,pct_ada_cc
0,307511,85.7,94.6,94.8,94.1,28.3


### 8.2 Data Mart versi 2: fitur riwayat diperkaya

**Alasan penambahan:** model di NB 04 (versi pertama) mencapai PR-AUC ±0,26. Analisis kesalahan menunjukkan agregasi riwayat di data mart versi 1 masih sangat sederhana (±25 kolom, semuanya rata-rata sepanjang masa). Padahal di credit scoring, **perilaku terbaru** dan **beban utang relatif** umumnya lebih prediktif (Siddiqi, 2017; Thomas et al., 2017). Versi 2 menambahkan:

| Tabel | Fitur baru | Alasan |
|---|---|---|
| Installments | Hari telat (DPD) rata-rata & maksimum, % telat > 30 hari, selisih bayar vs tagihan, perilaku **12 bulan terakhir** | Perilaku bayar terbaru lebih mencerminkan kondisi saat ini |
| Bureau | Rasio utang aktif terhadap plafon, jumlah kredit 12 bulan terakhir, kredit macet (*bad debt*), microloan, % bulan menunggak | Mengukur beban utang (temuan NB 03 Q12) |
| Previous application | Status pengajuan **terakhir**, jumlah penolakan 12 bulan terakhir, rasio kredit disetujui vs diajukan, uang muka, kelompok bunga | Pengajuan yang baru ditolak adalah sinyal risiko yang kuat |
| POS / cash | DPD dengan toleransi, % bulan menunggak 12 bulan terakhir, sisa cicilan | Perilaku terbaru |
| Kartu kredit | Utilisasi 6 bulan terakhir & maksimum, rasio pembayaran vs minimum, penarikan tunai ATM | Temuan NB 03 Q13 (utilisasi tinggi = berisiko) |

Query disimpan di `src/sql_v2.py` (SQL standar, kompatibel dengan BigQuery dan DuckDB). Data mart versi 1 tetap dipertahankan karena dipakai NB 03.

In [16]:
import sys; sys.path.append('.')
from src.sql_v2 import sql_agg_v2, sql_mart_v2
M = lambda name: T(DATASET_MART, name)

for name, sql in sql_agg_v2(R).items():
    run_to_table(sql, name)
run_to_table(sql_mart_v2(R, M, 'application_train'), 'mart_v2_train')
run_to_table(sql_mart_v2(R, M, 'application_test'),  'mart_v2_test')

checks_v2 = client.query(f'''
SELECT 'mart_v2_train' AS tabel, COUNT(*) AS baris, COUNT(DISTINCT SK_ID_CURR) AS nasabah_unik FROM {M('mart_v2_train')}
UNION ALL
SELECT 'mart_v2_test', COUNT(*), COUNT(DISTINCT SK_ID_CURR) FROM {M('mart_v2_test')}''').to_dataframe()
checks_v2['1_baris_per_nasabah'] = checks_v2.baris == checks_v2.nasabah_unik
display(checks_v2); assert checks_v2['1_baris_per_nasabah'].all()

OK agg_inst_v2           339,587 baris | 544 MB diproses | 8s
OK agg_bureau_v2         305,811 baris | 685 MB diproses | 9s
OK agg_prev_v2           338,857 baris | 118 MB diproses | 7s
OK agg_pos_v2            337,252 baris | 482 MB diproses | 8s
OK agg_cc_v2             103,558 baris | 293 MB diproses | 5s
OK mart_v2_train         307,511 baris | 392 MB diproses | 19s
OK mart_v2_test           48,744 baris | 187 MB diproses | 6s


,tabel,baris,nasabah_unik,1_baris_per_nasabah
0,mart_v2_train,307511,307511,True
1,mart_v2_test,48744,48744,True


## 9. Simpan salinan lokal Data Mart

Data mart diunduh sekali ke `data/processed/` sehingga NB 03 (memakai versi 1) dan NB 04 (memakai versi 2) **cukup membaca 1 file Parquet**, tanpa membaca ulang 8 CSV dan tanpa query berulang ke BigQuery.

In [17]:
from google.api_core.exceptions import NotFound

FORCE_DOWNLOAD_MART = False   # True = unduh ulang paksa

for name in ['mart_train', 'mart_test', 'mart_v2_train', 'mart_v2_test']:
    out      = PROCESSED_DIR / f'{name}.parquet'
    table_id = f'{PROJECT_ID}.{DATASET_MART}.{name}'
    tbl      = client.get_table(table_id)

    # Unduh hanya jika file lokal belum ada, atau tabel di BigQuery lebih baru dari file lokal
    perlu_unduh = FORCE_DOWNLOAD_MART or not out.exists() or tbl.modified.timestamp() > out.stat().st_mtime

    if perlu_unduh:
        t0 = time.time()
        df = client.list_rows(tbl).to_dataframe(progress_bar_type='tqdm')
        df.to_parquet(out, index=False)
        print(f'OK {name}: {df.shape[0]:,} baris × {df.shape[1]} kolom → {out}  ({time.time()-t0:.0f}s)')
    else:
        meta = pq.ParquetFile(out).metadata
        print(f'SKIP {name}: {meta.num_rows:,} baris × {meta.num_columns} kolom, file lokal sudah terbaru, dilewati')

SKIP mart_train: 307,511 baris × 146 kolom, file lokal sudah terbaru, dilewati
SKIP mart_test: 48,744 baris × 145 kolom, file lokal sudah terbaru, dilewati
Downloading: 100%|██████████|
OK mart_v2_train: 307,511 baris × 185 kolom → data\processed\mart_v2_train.parquet  (35s)
Downloading: 100%|██████████|
OK mart_v2_test: 48,744 baris × 184 kolom → data\processed\mart_v2_test.parquet  (9s)


## 10. Ringkasan

| Output | Lokasi | Dipakai oleh |
|---|---|---|
| 8 tabel Parquet (staging & backup) | `data/staging/` | Upload ulang ke BigQuery bila tabel kedaluwarsa |
| 8 tabel mentah | BigQuery `home_credit_raw` | Penelusuran data asli |
| 5 tabel agregasi + `mart_train` / `mart_test` (versi 1) | BigQuery `home_credit_mart` | NB 03, Power BI |
| 5 tabel agregasi v2 + `mart_v2_train` / `mart_v2_test` | BigQuery `home_credit_mart` | NB 04 (modeling) |
| Salinan data mart | `data/processed/` | NB 03 (EDA) & NB 04 (Modeling) |

**Kenapa Parquet + BigQuery, bukan salah satu?**
- **Parquet**: staging yang efisien untuk upload, backup lokal, dan bisa dipakai tanpa internet.
- **BigQuery**: data warehouse terpusat yang bisa diakses lewat SQL oleh notebook, Power BI, dan tim lain, serta skalabel bila data bertambah. Join dan agregasi tabel besar dikerjakan di sana.